# Practice 20 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [2]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: DateTime + MultiIndex
A coffee shop sales DataFrame has been created for you.

1. Convert `sale_date` to datetime. Extract `year` and `month` as new columns.
2. Add `days_ago` = days between `sale_date` and `2026-03-18`. Find mean and median with NumPy.
3. Set a `(city, product)` MultiIndex. Sort it.
4. Retrieve all rows for `'Seattle'`.
5. Use `pd.IndexSlice` to get all `'Latte'` rows across every city.

In [6]:
# Starter data — don't change this
sales = pd.DataFrame({
    'city':      ['Seattle', 'Seattle', 'Portland', 'Portland', 'Vancouver', 'Vancouver', 'Seattle', 'Portland'],
    'product':   ['Latte', 'Espresso', 'Latte', 'Cappuccino', 'Espresso', 'Latte', 'Cappuccino', 'Espresso'],
    'sale_date': ['2024-02-08', '2024-05-14', '2024-07-29', '2024-09-03',
                  '2024-11-17', '2025-01-22', '2025-04-10', '2025-07-05'],
    'units':     [42, 30, 55, 21, 38, 60, 18, 44],
    'revenue':   [189, 105, 247, 126, 133, 270, 108, 154],
})

# Your code here
sales['sale_date'] = pd.to_datetime(sales['sale_date'])
sales['year'] = sales['sale_date'].dt.year
sales['month'] = sales['sale_date'].dt.month
sales['days_ago'] = (pd.to_datetime('2026-03-18')- sales['sale_date']).dt.days
np.mean(sales['days_ago'])
np.median(sales['days_ago'])
sales = sales.set_index(['city','product']).sort_index()
sales.loc['Seattle']
idx = pd.IndexSlice
sales.loc[idx[:,'Latte'],:]

,,sale_date,units,revenue,year,month,days_ago
city,product,,,,,,
Portland,Latte,2024-07-29,55,247,2024,7,597
Seattle,Latte,2024-02-08,42,189,2024,2,769
Vancouver,Latte,2025-01-22,60,270,2025,1,420


---
## Level 2 — Transformations

### Task 2: .pipe() ✨ New

`.pipe()` lets you apply a function to a DataFrame as part of a method chain, so your transformations read left-to-right instead of inside-out.

```python
# Without .pipe() — hard to read, inside-out:
result = add_flag(normalize(drop_outliers(df)))

# With .pipe() — reads in the order things happen:
result = (
    df
    .pipe(drop_outliers)
    .pipe(normalize)
    .pipe(add_flag)
)
```

The function passed to `.pipe()` receives the DataFrame as its first argument. Extra arguments go after the function name:

```python
def add_margin(df, pct):
    df['margin'] = df['revenue'] * pct
    return df

df.pipe(add_margin, pct=0.2)
```

`.pipe()` is most useful when you have several custom cleaning or transformation steps to chain together cleanly.

---

A supermarket inventory DataFrame has been created for you. Three helper functions have also been defined.

1. Use `.pipe()` to apply all three functions in order: `flag_low_stock`, `apply_discount`, `add_value`.
2. How many items are flagged as low stock? Use the `low_stock` column.
3. What is the mean `discounted_price`? Use NumPy.
4. Which category has the highest total `stock_value`? Use groupby and `.idxmax()`.

In [36]:
# Starter data — don't change this
np.random.seed(3)
inventory = pd.DataFrame({
    'item':      ['Apples', 'Bread', 'Milk', 'Cheese', 'Pasta', 'Rice', 'Yogurt', 'Eggs'],
    'category':  ['Produce', 'Bakery', 'Dairy', 'Dairy', 'Dry Goods', 'Dry Goods', 'Dairy', 'Produce'],
    'price':     [1.20, 2.50, 1.80, 4.50, 1.10, 0.90, 2.20, 3.00],
    'stock':     np.random.randint(5, 120, size=8),
    'discount':  np.random.choice([0, 5, 10, 15], size=8),
})

def flag_low_stock(df, threshold=20):
    df['low_stock'] = df['stock'] < threshold
    return df

def apply_discount(df):
    df['discounted_price'] = df['price'] * (1 - df['discount'] / 100)
    return df

def add_value(df):
    df['stock_value'] = df['discounted_price'] * df['stock']
    return df

# Your code here

r = (
    inventory
    .pipe(flag_low_stock)
    .pipe(apply_discount)
    .pipe(add_value)
)

r['low_stock'].sum()
np.mean(r['discounted_price'])
r.groupby('category')['stock_value'].sum().idxmax()


'Dairy'

In [37]:
r = inventory.pipe(flag_low_stock).pipe(apply_discount).pipe(add_value)


### Task 3: stack() / unstack() / .xs()
A wide DataFrame of quarterly sales by region has been created for you.

1. Stack to long format. Store as `sales_long` (use `.to_frame('sales')`)
2. Add `log_sales` using NumPy
3. Add `strong`: `True` if `sales` > 500 (use `np.where`)
4. Use `.xs()` to extract all `'Q3'` rows across every region
5. Unstack back to wide format
6. Find the region with the highest mean sales using NumPy

In [13]:
# Starter data — don't change this
np.random.seed(17)
regions = pd.DataFrame({
    'region': ['North', 'South', 'East', 'West'],
    'Q1':     np.random.randint(200, 900, size=4),
    'Q2':     np.random.randint(200, 900, size=4),
    'Q3':     np.random.randint(200, 900, size=4),
    'Q4':     np.random.randint(200, 900, size=4),
}).set_index('region')

# Your code here
sales_long = regions.stack().to_frame('sales')
sales_long['log_sales'] = np.log(sales_long['sales'])
sales_long['strong'] = np.where(sales_long['sales']>500, True, False)
sales_long.xs('Q3',level=1)
w = sales_long.unstack()
rm = sales_long.groupby(level='region')['sales'].mean()
rm.idxmax()
#sales_long

'East'

### Task 4: .apply()
A gym membership DataFrame has been created for you.

1. Add `annual_fee`: row-wise `monthly_fee * 12` using `apply` with `axis=1`
2. Add `tier`: apply a lambda to `monthly_fee` — `'Premium'` if > 50, else `'Standard'`
3. Add `retention_risk` (row-wise): `'High'` if `visits_per_month < 4` **and** `months_active < 6`, `'Medium'` if either is true, else `'Low'`

In [ ]:
# Starter data — don't change this
np.random.seed(26)
members = pd.DataFrame({
    'member_id':       [f'M{i:03d}' for i in range(1, 9)],
    'monthly_fee':     [29.99, 69.99, 39.99, 59.99, 29.99, 79.99, 49.99, 69.99],
    'months_active':   np.random.randint(1, 48, size=8),
    'visits_per_month': np.random.randint(0, 20, size=8),
})

# Your code here
members['annual_fee'] = members.apply(lambda row: row['monthly_fee']*12, axis=1)
members['tier'] = members['monthly_fee'].apply(lambda x: 'Premium' if x>50
                                               else 'Standard')
members['retention_risk'] = members.apply(lambda row: 'High' if row['visits_per_month']<4 and row['months_active']<6
                                          else 'Medium' if row['visits_per_month'] < 4 or row['months_active']<6
                                          else 'Low', axis = 1)




### Task 5: .str + .rank()
A book catalog DataFrame has been created for you.

1. Add `title_upper`: titles in uppercase
2. Filter to books where `title` contains `'the'` (case-insensitive)
3. Extract the numeric part of `book_id` (e.g. `'B05'` → `'05'`) using `.str` slicing. Store as `num`.
4. Add `rating_rank`: rank by `rating`, highest first
5. Add `genre_rank`: rank within each `genre` by `rating`, highest first
6. Filter to the top-ranked book per genre (`genre_rank == 1`)

In [23]:
# Starter data — don't change this
np.random.seed(58)
books = pd.DataFrame({
    'book_id': [f'B{i:02d}' for i in range(1, 9)],
    'title':   ['The Great Gatsby', 'Dune', 'The Hobbit', 'Neuromancer',
                'The Road', 'Foundation', 'Beloved', 'The Alchemist'],
    'genre':   ['Fiction', 'Sci-Fi', 'Fantasy', 'Sci-Fi', 'Fiction', 'Sci-Fi', 'Fiction', 'Fantasy'],
    'rating':  np.round(np.random.uniform(3.5, 5.0, size=8), 2),
    'reviews': np.random.randint(500, 50000, size=8),
})

# Your code here
books['title_upper'] = books['title'].str.upper()
books[books['title'].str.contains('the', case=False)]
books['num'] = books['book_id'].str[1:]
books['rating_rank'] = books['rating'].rank(ascending=False)
books['genre_rank'] = books.groupby('genre')['rating'].rank(ascending=False)
books[books['genre_rank']==1] 

,book_id,title,genre,rating,reviews,title_upper,num,rating_rank,genre_rank
1,B02,Dune,Sci-Fi,4.18,36592,DUNE,02,3.0,1.0
2,B03,The Hobbit,Fantasy,4.24,8308,THE HOBBIT,03,2.0,1.0
4,B05,The Road,Fiction,4.36,24354,THE ROAD,05,1.0,1.0


---
## Level 3 — Aggregation

### Task 6: pd.merge() + pd.concat() + pivot_table
Two DataFrames have been created for you: `employees` and `performance`.

1. Inner join on `emp_id`
2. Add `dept_avg_score`: mean `score` per `dept` using `transform`
3. Add `above_dept_avg`: `True` if `score` > `dept_avg_score`
4. Two separate review DataFrames (`reviews_h1`, `reviews_h2`) have also been created. Concatenate them vertically, reset the index. Store as `reviews`.
5. Pivot `reviews`: mean `rating` by `dept` (rows) and `half` (columns)

In [29]:
# Starter data — don't change this
np.random.seed(82)
employees = pd.DataFrame({
    'emp_id': [f'E{i:02d}' for i in range(1, 9)],
    'name':   [f'Person_{i}' for i in range(1, 9)],
    'dept':   np.random.choice(['Engineering', 'Marketing', 'Finance'], size=8),
})

performance = pd.DataFrame({
    'emp_id': np.random.choice([f'E{i:02d}' for i in range(1, 9)], size=20),
    'score':  np.random.randint(50, 100, size=20),
    'month':  np.random.choice(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun'], size=20),
})

reviews_h1 = pd.DataFrame({
    'emp_id': np.random.choice([f'E{i:02d}' for i in range(1, 9)], size=8),
    'dept':   np.random.choice(['Engineering', 'Marketing', 'Finance'], size=8),
    'rating': np.random.randint(1, 6, size=8),
    'half':   'H1',
})

reviews_h2 = pd.DataFrame({
    'emp_id': np.random.choice([f'E{i:02d}' for i in range(1, 9)], size=8),
    'dept':   np.random.choice(['Engineering', 'Marketing', 'Finance'], size=8),
    'rating': np.random.randint(1, 6, size=8),
    'half':   'H2',
})

# Your code here

ij = pd.merge(
    employees, 
    performance,
    how = 'inner',
    on = 'emp_id'
)

ij['dept_avg_score'] = ij.groupby('dept')['score'].transform('mean')
ij['above_dept_avg'] = ij['score']> ij['dept_avg_score']
reviews = pd.concat(
    [reviews_h1, reviews_h2],
    ignore_index=True
)
pr = reviews.pivot_table(
    index = 'dept',
    columns='half',
    values='rating'
)
pr

half,H1,H2
dept,,
Engineering,3.5,1.000000
Finance,1.0,3.333333
Marketing,2.6,3.750000


---
## Level 4 — Real-world

### Task 7: Full Pipeline
Three DataFrames have been created for you: `products`, `orders`, and `customers`.
Three helper functions have also been defined — use `.pipe()` to apply them as part of your pipeline.

1. Convert `order_date` to datetime. Add `days_ago` = days from `order_date` to `2026-03-18`. Find mean and median with NumPy.
2. Inner join `orders` with `customers` on `customer_id`. Inner join with `products` on `product_id`. Drop nulls.
3. Use `.pipe()` to apply all three functions in order: `add_total`, `flag_large`, `add_category_avg`.
4. Use `.apply()` to add `priority` (row-wise): `'High'` if `order_total > 500` **and** `is_large` is `True`, `'Medium'` if either is true, else `'Low'`
5. Stack: pivot mean `order_total` by `region` (rows) and `category` (columns) using `pivot_table`. Stack the result.
6. Use `.xs()` to pull out just `'Electronics'` from the stacked result.
7. Find the correlation between `order_total` and `quantity` using NumPy

In [52]:
# Starter data — don't change this
np.random.seed(66)

products = pd.DataFrame({
    'product_id': [f'P{i:02d}' for i in range(1, 7)],
    'name':       ['Laptop', 'Phone', 'Tablet', 'Monitor', 'Keyboard', 'Headphones'],
    'category':   ['Electronics', 'Electronics', 'Electronics', 'Electronics', 'Accessories', 'Accessories'],
    'unit_price': [999, 699, 499, 349, 89, 149],
})

customers = pd.DataFrame({
    'customer_id': [f'C{i:02d}' for i in range(1, 9)],
    'region':      np.random.choice(['North', 'South', 'East', 'West'], size=8),
})

orders = pd.DataFrame({
    'order_id':    [f'O{i:03d}' for i in range(1, 25)],
    'customer_id': np.random.choice([f'C{i:02d}' for i in range(1, 9)], size=24),
    'product_id':  np.random.choice([f'P{i:02d}' for i in range(1, 7)], size=24),
    'quantity':    np.random.randint(1, 5, size=24),
    'order_date':  pd.date_range('2023-06-01', periods=24, freq='20D').strftime('%Y-%m-%d'),
})

def add_total(df):
    df['order_total'] = df['unit_price'] * df['quantity']
    return df

def flag_large(df, threshold=500):
    df['is_large'] = df['order_total'] > threshold
    return df

def add_category_avg(df):
    df['category_avg'] = df.groupby('category')['order_total'].transform('mean')
    return df

# Your code here
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['days_ago'] = (pd.to_datetime('2026-03-18')-orders['order_date']).dt.days
np.mean(orders['days_ago'])
np.median(orders['days_ago'])
ij = pd.merge(
    orders,
    customers,
    on='customer_id',
    how='inner'
)
ij2 = pd.merge(
    ij, 
    products, 
    on='product_id',
    how = 'inner'
).dropna()

res = (
    ij2
    .pipe(add_total)
    .pipe(flag_large)
    .pipe(add_category_avg)
)
res['priority'] = res.apply(lambda row: 'High' if row['order_total']>500 and row['is_large']
                            else 'Medium' if row['order_total']>500 or row['is_large']
                            else 'Low', axis = 1)
p = res.pivot_table(
    index = 'region',
    columns = 'category',
    values = 'order_total'
).stack().to_frame('order_total')
p.xs('Electronics',level='category')
np.corrcoef(res['order_total'], res['quantity'])

array([[1.        , 0.54142668],
       [0.54142668, 1.        ]])